#### silver-layer dataset;

- Assuming all datasets were properly masked for PII columns and for specific rows based on conditions like geo-locations,
next step is to create the silver-layer;

- silver-layer consists of datasets pre-AI models; or pre-Gold layer ready for visualization;


In [0]:
%sql
SELECT * FROM end_to_end_retail.db_bronze.tbb_bright_home_orders limit 5

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_bronze.tbb_customer_changes_daily limit 5

##### Applying mask

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def mask_value(value):
    if value is None:
        return None
    # Mask email addresses
    if '@' in value:
        local, domain = value.split('@', 1)
        masked_local = '*' * (len(local) - 2) if len(local) > 2 else '*' * len(local)
        masked_domain = '*' * (len(domain) - 4) + domain[-4:] if len(domain) > 4 else '*' * len(domain)
        return f"{masked_local}@{masked_domain}"
    # Mask names (first/last)
    if len(value) > 1:
        return '*' * (len(value) - 1)
    return '*' * len(value)

mask_value_udf = udf(mask_value, StringType())
spark.udf.register("mask_value", mask_value, StringType())

In [0]:
%sql
CREATE OR REPLACE TABLE end_to_end_retail.db_silver.tbs_bright_home_changes_orders_silver (
  order_id STRING,
  customer_id STRING PRIMARY KEY,
  region STRING,
  country STRING,
  city STRING,
  channel STRING,
  sku STRING,
  category STRING,
  qty STRING,
  unit_price STRING,
  discount_pct STRING,
  total_amount STRING,
  masked_email STRING,
  masked_first_name STRING,
  masked_last_name STRING,
  loyalty_tier STRING,
  operation STRING,
  source_subsidiary STRING,
  signup_date STRING,
  order_date STRING,
  timestamp STRING,
  order_timestamp STRING,
  ingestion_date_home_orders TIMESTAMP,
  ingestion_date_changes_daily TIMESTAMP,
  new_ingestion_date_orders_changes TIMESTAMP
);

INSERT INTO end_to_end_retail.db_silver.tbs_bright_home_changes_orders_silver
SELECT
  o.order_id,
  regexp_replace(o.customer_id, '^CUST_', '') AS customer_id,
  o.region,
  o.country,
  o.city,
  o.channel,
  o.sku,
  o.category,
  o.qty,
  o.unit_price,
  o.discount_pct,
  o.total_amount,
  mask_value(c.email) AS masked_email,
  mask_value(c.first_name) AS masked_first_name,
  mask_value(c.last_name) AS masked_last_name,
  c.loyalty_tier,
  c.operation,
  c.source_subsidiary,
  c.signup_date,
  o.order_date,
  c.timestamp,
  o.order_timestamp,
  o.ingestion_date as ingestion_date_home_orders,
  c.ingestion_date as ingestion_date_changes_daily,
  current_timestamp() as new_ingestion_date_orders_changes
FROM end_to_end_retail.db_bronze.tbb_bright_home_orders o
LEFT JOIN end_to_end_retail.db_bronze.tbb_customer_changes_daily c
  ON o.customer_id = c.customer_id

In [0]:
%sql
INSERT INTO end_to_end_retail.db_silver.tbs_bright_home_changes_orders_silver
VALUES
  ('BSH_ORD_20251101_0101', '12345', 'South America', 'Brazil', 'São Paulo', 'Online', 'SKU001', 'Electronics', '2', '500', '10', '900', '*****@****.com', '****', '****', 'Gold', 'Update', 'BR_SUB', '2026-01-01', '2026-02-01', '2026-02-07 10:00:00', '2026-02-07 10:00:00', current_timestamp(), current_timestamp(), current_timestamp()),
  ('BSH_ORD_20251101_0102', '67890', 'North America', 'USA', 'New York', 'Retail', 'SKU002', 'Home', '1', '200', '5', '190', '*****@****.com', '****', '****', 'Silver', 'Insert', 'US_SUB', '2026-01-02', '2026-02-02', '2026-02-07 11:00:00', '2026-02-07 11:00:00', current_timestamp(), current_timestamp(), current_timestamp()),
  ('BSH_ORD_20251101_0103', '54321', 'Europe', 'France', 'Paris', 'Online', 'SKU003', 'Fashion', '3', '150', '15', '382.5', '*****@****.com', '****', '****', 'Bronze', 'Delete', 'FR_SUB', '2026-01-03', '2026-02-03', '2026-02-07 12:00:00', '2026-02-07 12:00:00', current_timestamp(), current_timestamp(), current_timestamp());

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_silver.tbs_bright_home_changes_orders_silver where country like 'Brazil'

In [0]:
%sql
SELECT COUNT(*) FROM end_to_end_retail.db_silver.tbb_bright_home_changes_orders_silver

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_clicked_items limit 5

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 5

- SET spark.databricks.sql.dsv2.unique.enabled = true;


In [0]:
%sql
    
CREATE OR REPLACE TABLE end_to_end_retail.db_silver.tbs_customer_address_clicks_geoloc (
  customer_id STRING PRIMARY KEY,
  ship_to_address STRING,
  state STRING,
  city STRING,
  region STRING,
  district STRING,
  clicked_items STRING,
  lon DOUBLE,
  lat DOUBLE,
  valid_from STRING,
  valid_to STRING,
  insert_date DATE
);

INSERT INTO end_to_end_retail.db_silver.tbs_customer_address_clicks_geoloc
SELECT
  a.customer_id,
  g.ship_to_address,
  a.state,
  a.city,
  a.region,
  a.district,
  b.clicked_items,
  ROUND(CAST(g.lon AS DOUBLE), 2) AS lon,
  ROUND(CAST(g.lat AS DOUBLE), 2) AS lat,
  g.valid_from,
  g.valid_to,
  current_timestamp() as insert_date
FROM end_to_end_retail.db_landing.tbl_customer_address a
LEFT JOIN end_to_end_retail.db_landing.tbl_clicked_items b
  ON a.customer_id = b.customer_id
LEFT JOIN end_to_end_retail.db_landing.tbl_customer_geoloc g
  ON a.customer_id = g.customer_id
-- WHERE b.customer_id IS NOT NULL
-- WHERE b.customer_id IS NOT NULL

- Here a good dataset for clustering techniques, lat/lon, distance; specially clicks per region, worth finding insights 